# Acquisition round for student S6

Labels the 200 configurations chosen by `li3ocl/select_two_sided.py` with the teacher and trains two models with the architecture, loss and seed of S6 and the same label budget: **S6b** (two-sided acquisition) and **S6c** (control, 200 further equilibrium frames).

Inputs: `pool.xyz` and `test.xyz` from step 6 (Google Drive folder `li3ocl_step6/`), and `step8_bundle.zip` with `li3ocl_teacher.model` and `li3ocl_S6_two_sided_frames.npz`. Runtime: GPU. Output: `li3ocl_S6b.model` and `li3ocl_S6c.model`.

In [ ]:
!pip -q install mace-torch ase
import torch, time, json, os, subprocess, numpy as np
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
from google.colab import drive; drive.mount('/content/drive'); SRC = '/content/drive/MyDrive/li3ocl_step6'; OUT = '/content/drive/MyDrive/li3ocl_step8'; os.makedirs(OUT, exist_ok=True)
assert os.path.exists(f'{SRC}/pool.xyz') and os.path.exists(f'{SRC}/test.xyz'), 'li3ocl_step6/pool.xyz not found in Drive'
if not os.path.exists('li3ocl_S6_two_sided_frames.npz'):
    from google.colab import files
    up = files.upload()            # choose step8_bundle.zip
    os.system('unzip -o -q step8_bundle.zip')

In [ ]:
from ase import Atoms
from ase.io import read, write
from mace.calculators import MACECalculator
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
calc = MACECalculator(model_paths='li3ocl_teacher.model', device=DEV, default_dtype='float32')
d = np.load('li3ocl_S6_two_sided_frames.npz'); new = []
for x, src in zip(d['X'], d['source']):
    a = Atoms(numbers=d['numbers'], positions=x.astype(float), cell=d['cell'], pbc=True); a.wrap(); a.calc = calc
    e, f = a.get_potential_energy(), a.get_forces(); b = a.copy(); b.info['REF_energy'] = float(e); b.arrays['REF_forces'] = f; b.info['source'] = str(src); new.append(b)
E = np.array([a.info['REF_energy'] for a in new]); Fm = np.array([np.abs(a.arrays['REF_forces']).max() for a in new]); pool = read(f'{SRC}/pool.xyz', ':'); Ep = np.array([a.info['REF_energy'] for a in pool[:400]])
print(f'new labels: E {E.min():.1f}..{E.max():.1f} eV (equilibrium pool: {Ep.min():.1f}..{Ep.max():.1f}), largest force component {Fm.max():.1f} eV/A')
ok = np.isfinite(E) & (Fm < 50); print('usable labels:', int(ok.sum()), 'of', len(new)); new = [a for a, k in zip(new, ok) if k]
write('train_S6b.xyz', pool[:200] + new); write('train_S6c.xyz', pool[:400]); write(f'{OUT}/new_labels.xyz', new)

In [ ]:
REPORT = {}
for name in ('S6b', 'S6c'):
    if not os.path.exists(f'{OUT}/li3ocl_{name}.model'):
        t0 = time.time()
        cmd = (f"mace_run_train --name=li3ocl_{name} --train_file=train_{name}.xyz --valid_fraction=0.08 --test_file={SRC}/test.xyz --energy_key=REF_energy --forces_key=REF_forces "
               f"--E0s=average --model=MACE --hidden_irreps=16x0e --r_max=4.0 --num_interactions=2 --correlation=3 --batch_size=8 --valid_batch_size=8 --max_num_epochs=60 "
               f"--lr=0.01 --ema --ema_decay=0.99 --amsgrad --forces_weight=100 --energy_weight=1 --device={DEV} --default_dtype=float32 --seed=1 --save_cpu")
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print(name, 'exit', r.returncode, f'{(time.time() - t0) / 60:.0f} min'); print('\n'.join((r.stdout + r.stderr).splitlines()[-5:]), flush=True)
        os.system(f'cp li3ocl_{name}.model {OUT}/')
    c = MACECalculator(model_paths=f'{OUT}/li3ocl_{name}.model', device=DEV, default_dtype='float32'); test = read(f'{SRC}/test.xyz', ':'); dE, dF = [], []
    for a in test: b = a.copy(); b.calc = c; dE.append(b.get_potential_energy() - a.info['REF_energy']); dF.append(b.get_forces() - a.arrays['REF_forces'])
    dE, dF = np.array(dE), np.array(dF); REPORT[name] = dict(force_rmse_meVA=round(float(np.sqrt((dF ** 2).mean()) * 1e3), 1), energy_rmse_meV_cell=round(float((dE - dE.mean()).std() * 1e3), 1)); print(name, REPORT[name])
json.dump(REPORT, open(f'{OUT}/report.json', 'w'))
print('=========== SUMMARY ==========='); print(json.dumps(REPORT, indent=1)); print('==============================')
os.system(f'cd {OUT} && zip -q li3ocl_step8_results.zip li3ocl_S6b.model li3ocl_S6c.model new_labels.xyz report.json && cp li3ocl_step8_results.zip /content/')
from google.colab import files; files.download('/content/li3ocl_step8_results.zip')